# Topic Modelling & Analysis

In [3]:
import json
import random
import statistics
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from gensim.models.phrases import Phrases, Phraser
from wordcloud import WordCloud
from transformers import pipeline
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)

from sklearn.decomposition import (
    LatentDirichletAllocation,
    NMF
)

In [4]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DATA_PATH = DATA_DIR / "video_data_processed.json"
TOPICS_PROCESSED_VIDEO_DATA_PATH = DATA_DIR / "video_data_topics_processed.json"

RANDOM_STATE = 42

TOPIC_COUNTS = [2, 3, 4, 5, 8, 10]

TOP_WORDS = 10
TOP_MODELS_TO_INSPECT = 3
EXAMPLES_PER_TOPIC = 5

MIN_DF = 5
MAX_DF = 0.8

MAX_FEATURES = 10000

In [7]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

processed_comments = data["comments"]

print(f"Loaded {len(processed_comments)} processed comments\n")

Loaded 46808 processed comments



In [9]:
for i, row in enumerate(processed_comments[:5]):

    print("=" * 110)
    print(f"COMMENT {i+1}")

    print("\nORIGINAL:")
    print(row["comment_text"])

    print("\nTOPIC TEXT:")
    print(row["comment_text_topic"])

    print("\nTOKENS:")
    print(row["comment_tokens_topic"])

COMMENT 1

ORIGINAL:
The whole thing is so phoney STIFF UNNATURAL SMILES..some.unimaginative outfits...so pretenscious all this money spent could have fed the world!..

TOPIC TEXT:
whole thing phoney stiff unnatural smiles unimaginative outfits pretenscious money spent could fed world

TOKENS:
['whole', 'thing', 'phoney', 'stiff', 'unnatural', 'smiles', 'unimaginative', 'outfits', 'pretenscious', 'money', 'spent', 'could', 'fed', 'world']
COMMENT 2

ORIGINAL:
Free the CHILDREN

TOPIC TEXT:
free children

TOKENS:
['free', 'children']
COMMENT 3

ORIGINAL:
Thank you so much 🎉🎉🎉🎉🎉🎉🎉🎉🎉❤❤❤❤❤❤

TOPIC TEXT:
thank much

TOKENS:
['thank', 'much']
COMMENT 4

ORIGINAL:
Thanks, it's an extraordinary event with so many amazing gowns. ❤❤❤

TOPIC TEXT:
thanks extraordinary event many amazing gowns

TOKENS:
['thanks', 'extraordinary', 'event', 'many', 'amazing', 'gowns']
COMMENT 5

ORIGINAL:
Why that song?

TOPIC TEXT:
song

TOKENS:
['song']


### Helpers

In [10]:
# token preparation
tokenised_docs = [
    row["comment_tokens_topic"]
    for row in processed_comments
]

original_comments = [
    row["comment_text"]
    for row in processed_comments
]

In [11]:
# bigram detection
phrases_model = Phrases(
    tokenised_docs,
    min_count=5,
    threshold=10
)

bigram_model = Phraser(phrases_model)

documents_with_phrases = [
    bigram_model[doc]
    for doc in tokenised_docs
]

In [12]:
for i in range(5):

    print("=" * 100)
    print(f"DOCUMENT {i+1}")

    print("\nWITHOUT PHRASES:")
    print(tokenised_docs[i])

    print("\nWITH PHRASES:")
    print(documents_with_phrases[i])

DOCUMENT 1

WITHOUT PHRASES:
['whole', 'thing', 'phoney', 'stiff', 'unnatural', 'smiles', 'unimaginative', 'outfits', 'pretenscious', 'money', 'spent', 'could', 'fed', 'world']

WITH PHRASES:
['whole_thing', 'phoney', 'stiff', 'unnatural', 'smiles', 'unimaginative', 'outfits', 'pretenscious', 'money_spent', 'could', 'fed', 'world']
DOCUMENT 2

WITHOUT PHRASES:
['free', 'children']

WITH PHRASES:
['free', 'children']
DOCUMENT 3

WITHOUT PHRASES:
['thank', 'much']

WITH PHRASES:
['thank_much']
DOCUMENT 4

WITHOUT PHRASES:
['thanks', 'extraordinary', 'event', 'many', 'amazing', 'gowns']

WITH PHRASES:
['thanks', 'extraordinary', 'event', 'many', 'amazing', 'gowns']
DOCUMENT 5

WITHOUT PHRASES:
['song']

WITH PHRASES:
['song']


In [13]:
# converting tokens to text
documents_text = [
    " ".join(doc)
    for doc in documents_with_phrases
]

In [14]:
# custom stopwords
custom_stopwords = set(ENGLISH_STOP_WORDS).union({

    # conversational filler
    "like", "look", "looks", "looking",
    "really", "think", "know",
    "people", "dont", "didnt",
    "say", "said", "make", "made",
    "going", "got", "get",

    # generic praise
    "good", "best", "beautiful",
    "amazing", "great", "love",
    "pretty", "nice",

    # platform noise
    "video", "watch",

    # generic dataset noise
    "year", "years", "time",
    "thing", "stuff", "way",
    "actually", "lot",

    # met gala specific generic terms
    "met", "gala", "met_gala",

    # emojis / internet style
    "lol", "omg", "haha"
})